# Final — model and rule comparison testbed

This notebook evaluates several approaches for predicting the **Overall Situation**
on one raw dataset, under identical conditions, and reports which performs best.

**Dataset:** PAMAP2.

**Input:** raw accelerometer signal in 5-second windows (500 samples at 100 Hz) —
hand (x, y, z) and chest (x, y, z). The rule reads the windows directly; the models
read features extracted from them.

**Output classes — the 5 we predict and measure:** resting, still, walking, running,
in vehicle. Each window's label is derived from the dataset's activity annotation
(lying → resting; sitting / standing → still; walking / stairs / Nordic walking →
walking; running → running; car driving → in vehicle).

**Approaches compared:**

- **Deterministic rule** — the four orientation-invariant axes (activity, regularity,
  posture, locomotion) combined by fixed thresholds into a situation. No training.
- **Trained models** — standard classifiers (Logistic Regression, Random Forest,
  Gradient Boosting) on per-axis time- and frequency-domain features.

**Metrics** (identical windows, labels, and train/test split for every approach):

1. Accuracy on the data as recorded.
2. Accuracy after a random 3D rotation is applied to each window (a different device orientation).
3. Cost: number of features and training requirement.

## Load the dataset

Read the PAMAP2 Protocol and Optional files, keeping only the columns used here:
activity label (1), hand accelerometer (4–6), and chest accelerometer (21–23). The
Optional files are included because they contain the car-driving activity used for
the "in vehicle" class.

In [ ]:
import numpy as np, pandas as pd, glob, re

BASE  = "/home/voare/Documents/Synheart/Kinematics/Dataset/PAMAP2_data/PAMAP2_Dataset"
FILES = sorted(glob.glob(BASE+"/Protocol/*.dat")) + sorted(glob.glob(BASE+"/Optional/*.dat"))

# columns we use: 1 = activity label, 4-6 = hand accel xyz, 21-23 = chest accel xyz
USECOLS = [1, 4, 5, 6, 21, 22, 23]
COLNAMES = ["activity", "hand_x", "hand_y", "hand_z", "chest_x", "chest_y", "chest_z"]

data = []          # list of (subject_id, dataframe) — only the columns we need
for f in FILES:
    sid = int(re.search(r"subject(\d+)", f).group(1))
    df = pd.read_csv(f, sep=r"\s+", header=None, usecols=USECOLS)
    df.columns = COLNAMES
    data.append((sid, df))

print("files loaded:", len(data), " (Protocol + Optional)")
print("total rows  :", sum(len(df) for _, df in data))
data[0][1].head()

## The deterministic rule

The overall rule from `overall_state.ipynb`: the four orientation-invariant axes
(activity, regularity, posture, locomotion) combined by fixed thresholds into a
situation. No training.

In [ ]:
def hand_feats(w):                       # movement + repeat-rhythm from |accel|
    mag = np.sqrt((w**2).sum(axis=1)); m = mag - mag.mean()
    ac = np.correlate(m, m, mode="full")[len(m)-1:]
    rhythm = 0.0 if ac[0] == 0 else (ac/ac[0])[15:128].max()
    return mag.std(), float(rhythm)

def activity_state(mv):        return "sedentary" if mv < 0.5 else ("moderate" if mv < 7 else "vigorous")
def movement_regularity(mv, rg): return "calm" if mv < 1.0 else ("rhythmic" if rg > 0.40 else "erratic")
def locomotion(mv, rg):        return "stationary" if mv < 0.3 else ("on foot" if rg > 0.30 else "in vehicle")

def posture(chest_w, up):
    if np.sqrt((chest_w**2).sum(axis=1)).std() > 1.0: return "in motion"
    g = chest_w.mean(axis=0)
    tilt = np.degrees(np.arccos(np.clip(g @ up / (np.linalg.norm(g)*np.linalg.norm(up)+1e-9), -1, 1)))
    return "upright" if tilt < 45 else "lying"

def situation(act, reg, pos, loco):
    if pos == "lying":       return "resting (lying)"
    if loco == "on foot":    return "running" if act == "vigorous" else "walking"
    if loco == "in vehicle": return "in vehicle"
    if reg == "erratic":     return "restless"
    return "still"

def rule_predict(hand_w, chest_w, up):           # the rule on one window
    mv, rg = hand_feats(hand_w)
    return situation(activity_state(mv), movement_regularity(mv, rg), posture(chest_w, up), locomotion(mv, rg))

## Window the data and label it

Slide 5-second windows over each file, keep the activities that map to the five
classes, label each window by its situation, and skip windows containing gaps.
Posture uses a standing-gravity calibration taken per subject. These windows are
shared by the rule and the models.

In [ ]:
ACT_NAME = {1:"lying",2:"sitting",3:"standing",4:"walking",5:"running",
            7:"nordic_walk",11:"driving",12:"ascend_stairs",13:"descend_stairs"}
TO_SITUATION = {"lying":"resting (lying)","sitting":"still","standing":"still",
                "walking":"walking","nordic_walk":"walking","ascend_stairs":"walking",
                "descend_stairs":"walking","running":"running","driving":"in vehicle"}
CLASSES = ["resting (lying)", "still", "walking", "running", "in vehicle"]
WIN = 500

# standing-gravity calibration per subject (from their standing data), global fallback
subj_up = {}
for sid, df in data:
    st = df[df.activity == 3][["chest_x","chest_y","chest_z"]].to_numpy()
    st = st[~np.isnan(st).any(1)]
    if len(st): subj_up.setdefault(sid, st.mean(0))
global_up = np.mean(list(subj_up.values()), axis=0)

hands, chests, labels, subs = [], [], [], []
for sid, df in data:
    act   = df.activity.to_numpy()
    hand  = df[["hand_x","hand_y","hand_z"]].to_numpy()
    chest = df[["chest_x","chest_y","chest_z"]].to_numpy()
    change = np.where(np.diff(act) != 0)[0] + 1
    for s0, e0 in zip(np.r_[0, change], np.r_[change, len(act)]):
        name = ACT_NAME.get(act[s0])
        if name not in TO_SITUATION: continue
        for s in range(s0, e0 - WIN + 1, WIN):
            hw, cw = hand[s:s+WIN], chest[s:s+WIN]
            if np.isnan(hw).any() or np.isnan(cw).any(): continue
            hands.append(hw); chests.append(cw); labels.append(TO_SITUATION[name]); subs.append(sid)

hands  = np.array(hands); chests = np.array(chests)
labels = np.array(labels); subs = np.array(subs)
print("windows:", len(labels))
print(pd.Series(labels).value_counts().to_string())

## Run the rule and score it

Apply the rule to every window and compare to the true situation.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix

rule_pred = np.array([rule_predict(hands[i], chests[i], subj_up.get(subs[i], global_up))
                      for i in range(len(labels))])
rule_acc = accuracy_score(labels, rule_pred)

cm = confusion_matrix(labels, rule_pred, labels=CLASSES)
plt.figure(figsize=(7,6))
plt.imshow(cm, cmap="Blues")
plt.xticks(range(len(CLASSES)), CLASSES, rotation=30, ha="right")
plt.yticks(range(len(CLASSES)), CLASSES)
plt.xlabel("predicted"); plt.ylabel("true")
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max()/2 else "black")
plt.colorbar(label="windows")
plt.title(f"Deterministic rule — {rule_acc*100:.0f}% accurate")
plt.tight_layout(); plt.show()

print(f"deterministic rule accuracy: {rule_acc*100:.1f}%")